In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# STEP 1: IMPORT LIBRARIES
# Core libraries
import pandas as pd
import numpy as np

# Visualization (optional checks)
import matplotlib.pyplot as plt

# Preprocessing
from sklearn.preprocessing import StandardScaler


In [ ]:
# STEP 2: LOAD DATA
df = pd.read_csv('/content/drive/MyDrive/Dissertation Dataset/walmart_clean2.csv')

# Ensure correct date interpretation (DD/MM/YYYY format)
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)

# Sort data chronologically for time series analysis
df = df.sort_values('Date')

df.head()

,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment,lag_1,lag_4,rolling_4,Month,Week
0,1,2010-02-05,1643690.90,0,42.31,2.572,211.096358,8.106,NaN,NaN,NaN,2,6
1287,10,2010-02-05,2193048.75,0,54.34,2.962,126.442065,9.765,549731.49,606755.30,960813.63,2,6
5148,37,2010-02-05,536006.73,0,45.97,2.572,209.852966,8.554,272489.41,277137.86,349023.26,2,6
2288,17,2010-02-05,789036.02,0,23.11,2.666,126.442065,6.548,475770.14,471281.68,583455.58,2,6
4147,30,2010-02-05,465108.52,0,39.05,2.572,210.752605,8.324,534970.68,520632.80,507681.36,2,6


In [ ]:
# STEP 3: BASIC CLEANING
# Remove duplicates (if any)
df = df.drop_duplicates()

# Reset index
df = df.reset_index(drop=True)

# Quick check
print("Dataset shape after cleaning:", df.shape)


Dataset shape after cleaning: (6435, 13)


In [ ]:
# STEP 4: FEATURE ENGINEERING

df = df.sort_values(['Store', 'Date'])

df['lag_1'] = df.groupby('Store')['Weekly_Sales'].shift(1)
df['lag_4'] = df.groupby('Store')['Weekly_Sales'].shift(4)

df['rolling_mean_4'] = df.groupby('Store')['Weekly_Sales'] \
                        .rolling(4).mean().reset_index(0, drop=True)

df['rolling_std_4'] = df.groupby('Store')['Weekly_Sales'] \
                       .rolling(4).std().reset_index(0, drop=True)

df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Week'] = df['Date'].dt.isocalendar().week.astype(int)

df.head()

,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment,lag_1,lag_4,rolling_4,Month,Week,rolling_mean_4,rolling_std_4,Year
0,1,2010-02-05,1643690.90,0,42.31,2.572,211.096358,8.106,NaN,NaN,NaN,2,5,NaN,NaN,2010
59,1,2010-02-12,1641957.44,1,38.51,2.548,211.242170,8.106,1643690.90,NaN,NaN,2,6,NaN,NaN,2010
116,1,2010-02-19,1611968.17,0,39.93,2.514,211.289143,8.106,1641957.44,NaN,NaN,2,7,NaN,NaN,2010
174,1,2010-02-26,1409727.59,0,46.63,2.561,211.319643,8.106,1611968.17,NaN,1576836.03,2,8,1576836.025,112353.415114,2010
182,1,2010-03-05,1554806.68,0,46.50,2.625,211.350143,8.106,1409727.59,1643690.9,1554614.97,3,9,1554614.970,103135.002548,2010


In [ ]:
# STEP 5: CHECK AND HANDLE MISSING VALUES

print("Missing values before handling:")
print(df.isnull().sum())

df = df.dropna()

print("\nMissing values after handling:")
print(df.isnull().sum())

Missing values before handling:
Store               0
Date                0
Weekly_Sales        0
Holiday_Flag        0
Temperature         0
Fuel_Price          0
CPI                 0
Unemployment        0
lag_1              45
lag_4             180
rolling_4           3
Month               0
Week                0
rolling_mean_4    135
rolling_std_4     135
Year                0
dtype: int64

Missing values after handling:
Store             0
Date              0
Weekly_Sales      0
Holiday_Flag      0
Temperature       0
Fuel_Price        0
CPI               0
Unemployment      0
lag_1             0
lag_4             0
rolling_4         0
Month             0
Week              0
rolling_mean_4    0
rolling_std_4     0
Year              0
dtype: int64


In [ ]:
# STEP 6: FINAL DATA VALIDATION

print("Final dataset shape:", df.shape)

print("\nRemaining missing values:")
print(df.isnull().sum())

print("\nCheck data types:")
print(df.dtypes)

Final dataset shape: (6255, 16)

Remaining missing values:
Store             0
Date              0
Weekly_Sales      0
Holiday_Flag      0
Temperature       0
Fuel_Price        0
CPI               0
Unemployment      0
lag_1             0
lag_4             0
rolling_4         0
Month             0
Week              0
rolling_mean_4    0
rolling_std_4     0
Year              0
dtype: int64

Check data types:
Store                      int64
Date              datetime64[ns]
Weekly_Sales             float64
Holiday_Flag               int64
Temperature              float64
Fuel_Price               float64
CPI                      float64
Unemployment             float64
lag_1                    float64
lag_4                    float64
rolling_4                float64
Month                      int32
Week                       int64
rolling_mean_4           float64
rolling_std_4            float64
Year                       int32
dtype: object


In [ ]:
# STEP 7: FEATURE SELECTION

features = [
    'Store', 'Holiday_Flag',
    'Temperature', 'Fuel_Price', 'CPI', 'Unemployment',
    'lag_1', 'lag_4',
    'rolling_mean_4', 'rolling_std_4',
    'Year', 'Month', 'Week'
]

target = 'Weekly_Sales'

In [ ]:
# STEP 8: TRAIN-TEST SPLIT (TIME-SERIES SAFE)

unique_dates = sorted(df['Date'].unique())
split_index = int(len(unique_dates) * 0.8)
split_date = unique_dates[split_index]

train = df[df['Date'] < split_date]
test = df[df['Date'] >= split_date]

X_train = train[features]
y_train = train[target]

X_test = test[features]
y_test = test[target]

print("Split date:", split_date)
print("Training set size:", X_train.shape)
print("Test set size:", X_test.shape)

Split date: 2012-04-20 00:00:00
Training set size: (4995, 13)
Test set size: (1260, 13)


In [ ]:
#STEP 9: SCALING (FOR ML MODELS)
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [ ]:
# STEP 10: PREPARE DATA FOR PROPHET

prophet_df = df.groupby('Date', as_index=False)['Weekly_Sales'].sum()
prophet_df = prophet_df.rename(columns={'Date': 'ds', 'Weekly_Sales': 'y'})

prophet_train = prophet_df[prophet_df['ds'] < split_date]
prophet_test = prophet_df[prophet_df['ds'] >= split_date]

print("Prophet training data:", prophet_train.shape)
print("Prophet test data:", prophet_test.shape)

Prophet training data: (111, 2)
Prophet test data: (28, 2)


In [ ]:
df = df.drop(columns=['rolling_4'])

In [ ]:
# Save Phase 3 Cleaned Dataset
df.to_csv('/content/drive/MyDrive/Dissertation Dataset/walmart_phase3_final.csv', index=False)
print("Phase 3 dataset saved successfully.")

Phase 3 dataset saved successfully.


In [ ]:
# Check Dataset
df_check = pd.read_csv('/content/drive/MyDrive/Dissertation Dataset/walmart_phase3_final.csv')
print(df_check.columns)
print(df_check.shape)

Index(['Store', 'Date', 'Weekly_Sales', 'Holiday_Flag', 'Temperature',
       'Fuel_Price', 'CPI', 'Unemployment', 'lag_1', 'lag_4', 'Month', 'Week',
       'rolling_mean_4', 'rolling_std_4', 'Year'],
      dtype='object')
(6255, 15)
